# 01 – DQ Setup (run ONCE)
Creates the `dq` schema, all DQ tables, seeds the DQ dimensions and builds the date table.
Safe to re-run: tables use `IF NOT EXISTS`; only `dq_dimension` and `dim_date` are rebuilt.

In [ ]:
%run 00_dq_config

In [ ]:
# Step 1: schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DQ}")

In [ ]:
# Step 2: DQ dimensions (reference list - rebuilt on each run of this cell)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T_DIMENSION} (
    dimension_name        STRING,
    dimension_description STRING,
    sort_order            INT
) USING DELTA
""")

spark.sql(f"""
INSERT OVERWRITE {T_DIMENSION} VALUES
 ('Completeness', 'Required values are present',                         1),
 ('Validity',     'Values follow the expected format, type or domain',   2),
 ('Uniqueness',   'No unintended duplicate records',                     3),
 ('Consistency',  'Values agree across fields, tables or systems',        4),
 ('Accuracy',     'Values reflect the real-world truth / reference data', 5),
 ('Timeliness',   'Data is up to date',                                   6)
""")

In [ ]:
# Step 3: rule catalog
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T_CATALOG} (
    rule_id           INT,
    rule_name         STRING,
    rule_description  STRING,
    dimension_name    STRING,     -- must match dq_dimension.dimension_name
    entity_grain      STRING,     -- CUSTOMER / SITE / CONTACT ...
    base_sql          STRING,     -- population (denominator); must return record_key
    failure_condition STRING,     -- filter applied on base_sql (simple rules)
    failure_sql       STRING,     -- full failing-records query (complex rules); used only if failure_condition is NULL
    severity          STRING,     -- High / Medium / Low
    business_owner    STRING,
    is_active         BOOLEAN,
    rule_version      INT,
    created_ts        TIMESTAMP,
    modified_ts       TIMESTAMP
) USING DELTA
""")

In [ ]:
# Step 4: run header (one row per runner execution)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T_RUN} (
    run_id         STRING,
    run_date       DATE,
    start_ts       TIMESTAMP,
    end_ts         TIMESTAMP,
    status         STRING,    -- RUNNING / SUCCESS / PARTIAL / FAILED
    rules_executed INT,
    rules_errored  INT
) USING DELTA
""")

In [ ]:
# Step 5: rule run log (append - one row per rule per run; feeds trend + run-log page)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T_RUN_LOG} (
    run_id        STRING,
    run_date      DATE,
    run_ts        TIMESTAMP,
    rule_id       INT,
    rule_version  INT,
    total_count   BIGINT,
    failed_count  BIGINT,
    passed_count  BIGINT,
    pass_rate     DOUBLE,     -- per-rule display only; aggregate with counts in DAX
    status        STRING,     -- SUCCESS / FAILED
    error_message STRING,
    duration_sec  DOUBLE
) USING DELTA
""")

In [ ]:
# Step 6: failed records - history (append) and current (overwrite = business worklist)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T_FAILED_HIST} (
    run_id     STRING,
    run_date   DATE,
    rule_id    INT,
    record_key STRING
) USING DELTA
PARTITIONED BY (run_date)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T_FAILED_CURR} (
    rule_id           INT,
    record_key        STRING,
    run_id            STRING,
    run_date          DATE,
    first_failed_date DATE,
    days_failing      INT
) USING DELTA
""")

In [ ]:
# Step 7: date dimension (rebuilt on each run of this cell)
spark.sql(f"""
CREATE OR REPLACE TABLE {T_DATE} USING DELTA AS
SELECT
    d                                   AS date,
    year(d)                             AS year,
    quarter(d)                          AS quarter,
    month(d)                            AS month,
    date_format(d, 'MMM')               AS month_name,
    date_format(d, 'yyyy-MM')           AS year_month,
    weekofyear(d)                       AS week_of_year,
    dayofweek(d)                        AS day_of_week,
    date_format(d, 'EEE')               AS day_name
FROM (SELECT explode(sequence(DATE'2024-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d)
""")

In [ ]:
# Check
display(spark.sql(f"SHOW TABLES IN {DQ}"))